In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/DeeptraceReward


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/DeeptraceReward


In [ ]:
import os, shutil, requests

drive_path = '/content/drive/MyDrive/DeeptraceReward'

# === 3️⃣ Delete old folder (CAREFUL: this removes everything in it) ===
if os.path.exists(drive_path):
    print(f"⚠️ Deleting existing folder: {drive_path}")
    shutil.rmtree(drive_path)
os.makedirs(drive_path, exist_ok=True)
os.chdir(drive_path)
print(f"📁 Now working in: {os.getcwd()}")


⚠️ Deleting existing folder: /content/drive/MyDrive/DeeptraceReward
📁 Now working in: /content/drive/MyDrive/DeeptraceReward


In [ ]:
urls = [
    "https://huggingface.co/datasets/DeeptraceReward/RewardData/resolve/main/all_data.json",
    "https://huggingface.co/datasets/DeeptraceReward/RewardData/resolve/main/real_videos.zip",
    "https://huggingface.co/datasets/DeeptraceReward/RewardData/resolve/main/videos.zip.001",
    "https://huggingface.co/datasets/DeeptraceReward/RewardData/resolve/main/videos.zip.002",
    "https://huggingface.co/datasets/DeeptraceReward/RewardData/resolve/main/videos.zip.003",
    "https://huggingface.co/datasets/DeeptraceReward/RewardData/resolve/main/videos.zip.004"
]

# === 5️⃣ Download each file safely in streaming mode ===
for url in urls:
    filename = url.split("/")[-1]
    print(f"\n⬇️ Downloading {filename} ...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        size = 0
        with open(filename, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
                if chunk:
                    f.write(chunk)
                    size += len(chunk)
                    if size % (500 * 1024 * 1024) < 1024 * 1024:  # every ~500 MB
                        print(f"   {size/1024/1024:.1f} MB downloaded...", end="\r")
    print(f"✅ Finished {filename} ({size/1024/1024:.2f} MB)")

# === 6️⃣ Verify all files ===
print("\n📦 Final contents:")
!ls -lh


⬇️ Downloading all_data.json ...
✅ Finished all_data.json (10.37 MB)

⬇️ Downloading real_videos.zip ...
✅ Finished real_videos.zip (3880.14 MB)

⬇️ Downloading videos.zip.001 ...
✅ Finished videos.zip.001 (5120.00 MB)

⬇️ Downloading videos.zip.002 ...
✅ Finished videos.zip.002 (5120.00 MB)

⬇️ Downloading videos.zip.003 ...
✅ Finished videos.zip.003 (5120.00 MB)

⬇️ Downloading videos.zip.004 ...
✅ Finished videos.zip.004 (994.89 MB)

📦 Final contents:
total 20G
-rw------- 1 root root  11M Nov 10 09:18 all_data.json
-rw------- 1 root root 3.8G Nov 10 09:19 real_videos.zip
-rw------- 1 root root 5.0G Nov 10 09:21 videos.zip.001
-rw------- 1 root root 5.0G Nov 10 09:23 videos.zip.002
-rw------- 1 root root 5.0G Nov 10 09:27 videos.zip.003
-rw------- 1 root root 995M Nov 10 09:27 videos.zip.004


In [ ]:
# ============================================================
# Verify Hugging Face dataset files & safely extract them
# ============================================================
import os
from google.colab import drive

# --- 1️⃣ Mount Google Drive ---
drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/DeeptraceReward'
os.chdir(drive_path)

# --- 2️⃣ Expected file sizes (in bytes) ---
expected_sizes = {
    "videos.zip.001": 5_370_000_000,
    "videos.zip.002": 5_370_000_000,
    "videos.zip.003": 5_370_000_000,
    "videos.zip.004": 1_040_000_000,
    "real_videos.zip": 4_070_000_000,
    "all_data.json": 10_900_000
}

# --- 3️⃣ Verify presence and size ---
print("🔍 Checking files in:", drive_path)
all_ok = True
for fname, expected in expected_sizes.items():
    if not os.path.exists(fname):
        print(f"❌ Missing: {fname}")
        all_ok = False
        continue
    actual = os.path.getsize(fname)
    diff = abs(actual - expected)
    pct = diff / expected * 100
    status = "✅ OK" if pct < 1 else "⚠️ Size mismatch"
    print(f"{status:15s} {fname:15s}  {actual/1e9:6.2f} GB (expected ~{expected/1e9:.2f} GB)")
    if pct >= 1:
        all_ok = False

# --- 4️⃣ Proceed only if everything is complete ---
if not all_ok:
    print("\n⚠️ Some files are incomplete or missing.")
    print("➡️ Wait for Drive to finish syncing or re-download the missing parts.")
else:
    print("\n✅ All files verified — ready to extract!\n")
    # --- 5️⃣ Extract multi-part archive ---
    !sudo apt-get install -y p7zip-full
    !7z x videos.zip.001 -o./videos_extracted -y
    print("\n📂 Extraction complete. Contents of videos_extracted:")
    !ls -lh videos_extracted | head


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Checking files in: /content/drive/MyDrive/DeeptraceReward
✅ OK            videos.zip.001     5.37 GB (expected ~5.37 GB)
✅ OK            videos.zip.002     5.37 GB (expected ~5.37 GB)
✅ OK            videos.zip.003     5.37 GB (expected ~5.37 GB)
✅ OK            videos.zip.004     1.04 GB (expected ~1.04 GB)
✅ OK            real_videos.zip    4.07 GB (expected ~4.07 GB)
✅ OK            all_data.json      0.01 GB (expected ~0.01 GB)

✅ All files verified — ready to extract!

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs AMD EPYC 7B12 (830F10),ASM

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/DeeptraceReward
!nohup 7z x real_videos.zip -o./real_videos -y > real_unzip.log 2>&1 &

MessageError: Error: credential propagation was unsuccessful